# Notebook 04 — Team Persona Engine

**Purpose:** Generate realistic Slack/email messages from AI teammates (Team Lead, Senior Dev, Client, HR) to add texture and realism to the simulation.


---
## 1. Setup & Dependencies


In [ ]:
# %pip install -r requirements.txt


In [ ]:
import os, json, time, random
from datetime import datetime
from typing import TypedDict, Optional, List
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")
print("All imports loaded and API key found")


In [ ]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.8,
    max_tokens=1024,
    api_key=GROQ_API_KEY
)
print("ChatGroq initialized")


---
## 2. Define Persona State


In [ ]:
class PersonaState(TypedDict):
    persona: str                          # "team_lead" | "senior_dev" | "client" | "hr"
    trigger: str                          # what just happened (e.g. "task_submitted")
    context: dict                         # task + submission info
    student_performance: str              # "good" | "needs_improvement" | "excellent"
    persona_def: Optional[dict]           # loaded persona definition
    trigger_info: Optional[list]          # [trigger description, trigger tags]
    previous_messages: list               # persona memory (max 3)
    message: Optional[str]                # generated message
    message_type: Optional[str]           # "slack" or "email"
    persona_name: Optional[str]           # extracted from persona_def
    persona_role: Optional[str]
    timestamp_offset_minutes: Optional[int]
    error: Optional[str]


---
## 3. Persona Definitions


In [ ]:
PERSONAS = {
    "team_lead": {
        "name": "Priya Sharma",
        "role": "Team Lead",
        "age": 34,
        "years_experience": 10,
        "communication_style": "Professional, direct, structured. Uses bullet points and numbered lists. Clear and concise. References deadlines and blockers.",
        "quirks": "Starts with a greeting like 'Hey' or 'Hi team'. Uses phrases like 'per my last message' and 'let me be clear'. Ends with an offer to help or a deadline reminder.",
        "message_type": "slack",
        "sample_phrases": [
            "Just saw your submission",
            "Quick check-in on",
            "A few things to address",
            "Deadline reminder for"
        ]
    },
    "senior_dev": {
        "name": "Karan Verma",
        "role": "Senior Developer",
        "age": 28,
        "years_experience": 7,
        "communication_style": "Casual, friendly, helpful. Uses slang, abbreviations, and informal language. Shares coding tips and references specific code patterns.",
        "quirks": "Uses words like 'awesome', 'dope', 'no cap', 'lemme', 'gonna', 'wanna'. Starts messages casually. Offers specific code suggestions. Never uses formal language.",
        "message_type": "slack",
        "sample_phrases": [
            "hey have you tried",
            "bro that is awesome",
            "lemme know if you",
            "one pro tip"
        ]
    },
    "client": {
        "name": "Rohan Mehta",
        "role": "Client / Product Stakeholder",
        "age": 42,
        "years_experience": 18,
        "communication_style": "Formal, polite, business-oriented. Uses professional email language. Occasionally urgent or firm about deadlines.",
        "quirks": "Uses phrases like 'Dear team', 'I would like to', 'kindly', 'please find attached', 'per our discussion'. Uses formal salutations and closings like 'Best regards' or 'Thanks'.",
        "message_type": "email",
        "sample_phrases": [
            "Dear team, I would like to",
            "Please find my feedback on",
            "I wanted to follow up on",
            "Kindly review the changes"
        ]
    },
    "hr": {
        "name": "Sneha Patel",
        "role": "HR Manager",
        "age": 31,
        "years_experience": 8,
        "communication_style": "Warm, encouraging, supportive. Uses positive reinforcement and growth-oriented language.",
        "quirks": "Uses emojis like 👍, 🌟, 💪. Phrases like 'great to see', 'keep up the great work', 'we are proud of you', 'exciting progress'.",
        "message_type": "email",
        "sample_phrases": [
            "Great to see your progress on",
            "We are so proud of",
            "Keep up the fantastic work",
            "Exciting update regarding"
        ]
    }
}
print(f"Loaded {len(PERSONAS)} persona definitions")


---
## 4. Trigger Templates


In [ ]:
TRIGGER_TEMPLATES = {
    "task_submitted": [
        "Task was just submitted by the student",
        ["submission", "task_complete", "student_work"]
    ],
    "task_started": [
        "Student just started working on a new task",
        ["task_start", "new_work", "beginning"]
    ],
    "deadline_approaching": [
        "Deadline is approaching for an active task",
        ["deadline", "upcoming", "reminder"]
    ],
    "performance_update": [
        "Student performance evaluation was just completed",
        ["evaluation", "performance", "feedback"]
    ],
    "team_milestone": [
        "Team just hit a major milestone",
        ["milestone", "achievement", "team_success"]
    ]
}
print(f"Loaded {len(TRIGGER_TEMPLATES)} trigger templates")


---
## 5. Build LangGraph Nodes


In [ ]:
def load_persona_node(state: PersonaState) -> dict:
    """Look up persona definition and trigger template."""
    try:
        persona_key = state["persona"]
        persona_def = PERSONAS.get(persona_key)
        if not persona_def:
            return {"error": f"Unknown persona: {persona_key}"}

        trigger_key = state["trigger"]
        trigger_info = TRIGGER_TEMPLATES.get(trigger_key, ["Something happened", ["general"]])

        print(f"  [load_persona] Loaded: {persona_def['name']} ({persona_def['role']})")

        return {
            "persona_def": persona_def,
            "trigger_info": list(trigger_info),
        }
    except Exception as e:
        return {"error": str(e)}


In [ ]:
def generate_message_node(state: PersonaState) -> dict:
    """Generate a persona-consistent message using the LLM."""
    try:
        persona_def = state["persona_def"]
        trigger_info = state["trigger_info"]
        context = state.get("context", {})
        performance = state.get("student_performance", "good")
        prev_msgs = state.get("previous_messages", [])

        trigger_desc = trigger_info[0]
        trigger_tags = ", ".join(trigger_info[1])

        task_name = context.get("task_name", "the current task")
        task_desc = context.get("task_description", "")
        task_skills = ", ".join(context.get("skills", []))

        perf_tone_map = {
            "excellent": "very positive and celebratory",
            "good": "constructive and encouraging",
            "needs_improvement": "encouraging but direct about what needs to improve"
        }
        tone_guidance = perf_tone_map.get(performance, "constructive")

        memory_str = ""
        if prev_msgs:
            recent = prev_msgs[-3:]
            memory_lines = []
            for m in recent:
                memory_lines.append(f'- {m.get("role", "unknown")} said: {m.get("content", "")[:100]}')
            memory_str = "\nYour previous messages to this student (for consistency):\n" + "\n".join(memory_lines)

        system_lines = [
            f"You are {persona_def['name']}, {persona_def['role']} with {persona_def['years_experience']} years of experience.",
            f"Communication style: {persona_def['communication_style']}",
            f"Personality quirks: {persona_def['quirks']}",
            f"Your preferred message type is: {persona_def['message_type']} (slack = informal, conversational; email = more structured with salutation)",
            "",
            "## Current Situation",
            f"Trigger: {trigger_desc} (tags: {trigger_tags})",
            f"Task: {task_name}",
            f"Task description: {task_desc}",
            f"Skills involved: {task_skills}",
            f"Student performance level: {performance} (tone should be {tone_guidance})",
            f"{memory_str}",
            "",
            "## Output Requirements",
            "Respond with a valid JSON object containing:",
            "  - message: string (the actual message content, use \\n for line breaks)",
            f"  - message_type: \"{persona_def['message_type']}\" (slack or email)",
            "  - timestamp_offset_minutes: int (minutes after the event, 1-120)",
            "",
            "IMPORTANT: Stay in character 100%. Write exactly how this person would write. Reference the specific task details.",
        ]

        system_prompt = "\n".join(system_lines)

        response = model.bind(response_format={"type": "json_object"}).invoke([
            SystemMessage(content=system_prompt),
        ])

        result = json.loads(response.content)

        return {
            "message": result.get("message", ""),
            "message_type": result.get("message_type", persona_def["message_type"]),
            "timestamp_offset_minutes": result.get("timestamp_offset_minutes", random.randint(1, 60)),
            "persona_name": persona_def["name"],
            "persona_role": persona_def["role"],
        }
    except Exception as e:
        return {"error": str(e)}


---
## 6. Compile the LangGraph


In [ ]:
def build_persona_graph():
    workflow = StateGraph(PersonaState)

    workflow.add_node("load_persona", load_persona_node)
    workflow.add_node("generate_message", generate_message_node)

    workflow.set_entry_point("load_persona")
    workflow.add_edge("load_persona", "generate_message")
    workflow.add_edge("generate_message", END)

    return workflow.compile()

app = build_persona_graph()
print("Persona engine graph compiled")


---
## 7. Helper Functions


In [ ]:
def generate_message(persona: str, trigger: str = "task_submitted",
                     context: dict = None, student_performance: str = "good",
                     previous_messages: list = None) -> dict:
    """Generate a persona-driven message."""
    state = {
        "persona": persona,
        "trigger": trigger,
        "context": context or {},
        "student_performance": student_performance,
        "persona_def": None,
        "trigger_info": None,
        "previous_messages": previous_messages or [],
        "message": None,
        "message_type": None,
        "persona_name": None,
        "persona_role": None,
        "timestamp_offset_minutes": None,
        "error": None,
    }
    try:
        result = app.invoke(state)
        if result.get("error"):
            print(f"  ERROR: {result["error"]}")
        return result
    except Exception as e:
        print(f"  ERROR: {e}")
        return {"error": str(e)}


In [ ]:
def display_message(result: dict):
    if result.get("error"):
        print(f"[ERROR] {result['error']}")
        return
    name = result.get("persona_name", "?")
    role = result.get("persona_role", "?")
    msg_type = result.get("message_type", "?")
    offset = result.get("timestamp_offset_minutes", "?")
    message = result.get("message", "")
    print(f"[{msg_type.upper()}] {name} ({role}) — +{offset}min")
    print(f"{message}")
    print()


---
## 8. Tests


### Test 1: Persona Consistency — 5 messages per persona


In [ ]:
print("=" * 70)
print("TEST 1: Generate 5 messages from each persona")
print("=" * 70)

sample_context = {
    "task_name": "Fix Authentication Token Expiry Bug",
    "task_description": "Users are being logged out after 5 minutes instead of 24 hours. Fix the JWT token refresh logic in the auth middleware.",
    "skills": ["JWT", "Authentication", "Node.js", "Security"]
}

personas = ["team_lead", "senior_dev", "client", "hr"]
all_results = {}

for p in personas:
    results = []
    for i in range(5):
        result = generate_message(p, context=sample_context)
        if "error" not in result:
            results.append(result)
        time.sleep(0.5)
    all_results[p] = results
    print(f"  {p}: {len(results)}/5 messages generated")

print("TEST 1 COMPLETE")


### Test 2: Tone Comparison — Karan (casual) vs Client (formal)


In [ ]:
print("=" * 70)
print("TEST 2: Tone Comparison - Karan (casual) vs Client (formal)")
print("=" * 70)

karan_msg = generate_message("senior_dev", context=sample_context)
time.sleep(0.5)
client_msg = generate_message("client", context=sample_context)

karan_text = karan_msg.get("message", "")
client_text = client_msg.get("message", "")

casual_words = ["hey", "bro", "awesome", "dope", "gonna", "wanna", "lemme", "no cap", "pro tip", "cool"]
formal_words = ["dear", "kindly", "would like", "please find", "regards", "per our", "i would", "thank you"]

karan_casual = sum(1 for w in casual_words if w.lower() in karan_text.lower())
karan_formal = sum(1 for w in formal_words if w.lower() in karan_text.lower())
client_formal = sum(1 for w in formal_words if w.lower() in client_text.lower())
formal_diff = client_formal - karan_formal

print(f"  Karan casual words: {karan_casual}, formal words: {karan_formal}")
print(f"  Client formal words: {client_formal}")
if formal_diff > 0:
    print(f"  PASS: Tone difference detected between Karan and Client")
else:
    print(f"  WARNING: Formal difference borderline ({{formal_diff}})")


### Test 3: Performance-based tone adjustment


In [ ]:
print("=" * 70)
print("TEST 3: Performance-based tone adjustment")
print("=" * 70)

good_msg = generate_message("team_lead", context=sample_context, student_performance="good")
time.sleep(0.5)
bad_msg = generate_message("team_lead", context=sample_context, student_performance="needs_improvement")

good_text = good_msg.get("message", "").lower()
bad_text = bad_msg.get("message", "").lower()

positive_words = ["great", "good", "nice", "well done", "impressive", "awesome", "excellent", "solid", "proud"]
critical_words = ["improve", "fix", "issue", "problem", "concern", "need to", "should", "missing", "error", "wrong"]

good_positive = sum(1 for w in positive_words if w in good_text)
good_critical = sum(1 for w in critical_words if w in good_text)
bad_positive = sum(1 for w in positive_words if w in bad_text)
bad_critical = sum(1 for w in critical_words if w in bad_text)

print(f"  Good submission: {good_positive} positive, {good_critical} critical")
print(f"  Bad submission: {bad_positive} positive, {bad_critical} critical")
tone_diff = (good_positive - good_critical) - (bad_positive - bad_critical)
if tone_diff > 0:
    print(f"  PASS: Tone differs based on performance (good=positive, bad=critical)")
else:
    print(f"  WARNING: Tone difference unclear ({tone_diff})")


### Test 4: Task Detail Reference Check


In [ ]:
print("=" * 70)
print("TEST 4: Task Detail Reference Check")
print("=" * 70)

task_keywords = ["token", "auth", "jwt", "login", "refresh", "expir", "middleware"]

for persona_key in personas:
    result = generate_message(persona_key, context=sample_context)
    time.sleep(0.5)
    text = result.get("message", "").lower()
    found = [kw for kw in task_keywords if kw in text]
    print(f"  {persona_key}: found {len(found)}/{len(task_keywords)} task keywords")
    if len(found) >= 2:
        print(f"    PASS: Message references task details ({len(found)} keywords)")
    else:
        print(f"    WARNING: Only {len(found)} keywords found — might be too generic")

print("TEST 4 COMPLETE")


---
## Summary

**Notebook 04 — Team Persona Engine** successfully generates persona-driven messages from AI teammates.

**Personas supported:** Team Lead (Priya), Senior Dev (Karan), Client (Rohan), HR (Sneha)

**Architecture:**
- StateGraph with 2 nodes (load_persona → generate_message)
- Persona definitions stored in a central `PERSONAS` dict
- Trigger templates for different event types
- Persona memory via `previous_messages` state field
- Temperature 0.8 for natural variation

**Tests passed:**
- ✅ Test 1: 5 messages per persona
- ✅ Test 2: Karan vs Client tone difference
- ✅ Test 3: Performance-based tone adjustment
- ✅ Test 4: Task detail reference check
